# Chapter 19: Inference Optimization
## Topic 2: Prompt Caching — What's Cacheable in Your Pipeline

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 1 established that input tokens are billed on every single API call, including content that's often identical across many different calls — this project's system prompt (Chapter 3, Chapter 9 Topic 6, Chapter 13's additions), tool schemas (Chapter 10, sent unchanged on every call per Topic 1's own real, computed 400-token estimate), and often substantial portions of a knowledge base's retrieved content, don't actually change from one email to the next, yet Topic 1's naive accounting billed them fresh every time. **Prompt caching** is the specific mechanism this topic introduces to stop paying full price for genuinely repeated, unchanged content.
- The core mechanism, precisely: Claude's API supports marking specific portions of a request as cacheable — on the first request including that content, it's processed and cached (at a real, documented cost premium for the initial cache write, roughly 1.25x the standard input rate for a short-lived cache), but on every subsequent request within the cache's validity window that includes the *exact same* cached content, that portion is billed at a dramatically reduced rate — roughly 10% of the standard input cost (a 90% discount), directly reducing Topic 1's own real, computed per-email cost for exactly the content that doesn't actually vary.
- Why this project's specific pipeline is unusually well-suited to caching: Topic 1's own real breakdown showed system prompt (350 tokens) and tool schemas (400 tokens) together account for roughly 750 of the estimated 1,103 total input tokens per email — a substantial majority of input tokens that are, by construction, identical across every single email this project processes, making this project's actual, real token composition a strong, concrete case for caching's benefit, not merely a theoretical one.


### 2. Internal Working — Step by Step

**How prompt caching actually works mechanically, and what determines whether it helps for a specific piece of content, step by step:**

1. **Identify content that's genuinely stable across many requests** — this project's system prompt and tool schemas are the clearest, strongest candidates, since Chapter 10 Topic 2's own message mechanics confirm these are sent, unchanged, with every single API call this project's agent makes, regardless of which specific email is being processed.
2. **Mark this stable content as cacheable in the API request** — the specific mechanism (a `cache_control` parameter on the relevant content block, in Claude's actual API) tells the API this exact content should be cached for potential reuse on a subsequent call, rather than treated as entirely fresh, one-off input every time.
3. **The first request including this content pays a real, one-time cache-write premium** (roughly 1.25x the standard input rate for the shorter-lived, 5-minute cache option) — this is a genuine, upfront cost, not free, meaning caching only pays off once this content is actually reused across multiple subsequent requests within the cache's validity window, not for genuinely one-off content.
4. **Every subsequent request within the cache's validity window that includes this exact same content is billed at the discounted cache-hit rate** (roughly 10% of standard input cost) — given this project's real, stated production volume (Topic 1's 8,000-12,000 emails/day), the system prompt and tool schemas would be reused across an enormous number of subsequent requests within any reasonable cache-validity window, meaning the one-time cache-write premium is amortized across a genuinely large number of cache hits, producing substantial real net savings.
5. **Content that genuinely varies per request — the specific customer email, and (depending on the specific query) some or all of the RAG-retrieved context — is generally not a good caching candidate**, since caching's benefit specifically depends on the *same* content being reused across multiple calls, which a customer's own, unique email content structurally cannot be.


### 3. How This Is Implemented in This Project

- This project's system prompt (Chapter 3's original instructions, extended by Chapter 9 Topic 6's RAG-specific additions and Chapter 13's confidence-framing additions) is the single clearest, highest-value caching candidate — genuinely identical across every email this project processes, and, per Topic 1's own real estimate, a meaningful fraction (roughly 32%) of total input tokens per email.
- This project's tool schemas (Chapter 10's four real tools — `validate_fd_reference`, `search_knowledge_base`, `lookup_product_catalog`, `check_sender_history`) are the second clear caching candidate, equally stable across every request and, per Topic 1's own real estimate, an even larger fraction (roughly 36%) of total input tokens per email.
- This project's RAG-retrieved knowledge-base content (Chapter 7's chunks) presents a more nuanced caching opportunity — while the *specific* chunks retrieved vary by query, the underlying knowledge base itself (Chapter 7's FAQ, SOP, and policy documents) is relatively stable over time (updated only when policy genuinely changes, per Chapter 9 Topic 8's own re-ingestion discussion), meaning a smart caching strategy could cache commonly-retrieved chunks specifically (the penalty policy, senior-citizen rate information, and other frequently-relevant content this project's own real data shows recurring across many different emails) even though this requires more careful, query-pattern-aware design than the straightforward system-prompt and tool-schema caching cases.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Caching only pays off when cached content is actually reused enough times to amortize the initial cache-write premium** — for content that changes on nearly every request (this project's own actual customer email content, definitionally unique per email), caching would only add the write premium's cost with no subsequent hit-rate benefit to offset it, making it actively counterproductive to attempt caching this kind of genuinely per-request-unique content.
- **Cache validity windows (Chapter 19's own real, documented options — a shorter 5-minute window at a lower write premium, or a longer 1-hour window at a higher write premium) need to be matched to this project's actual request pattern** — given this project's real, stated volume (8,000-12,000 emails/day, meaning frequent, near-continuous request traffic), even the shorter 5-minute cache window would likely see the system prompt and tool schemas reused many times before expiring, making the shorter, cheaper cache-write option a reasonable, cost-effective default for this project's specific traffic pattern.
- **A system prompt or tool schema that changes frequently (during active development, following Chapter 15 Topic 5's own version-tracking discussion) reduces caching's practical benefit**, since each change effectively invalidates the existing cache, requiring a fresh cache-write premium to be paid again — this creates a genuine tension between rapid, evidence-based iteration (this notebook series' own repeated recommendation) and caching's benefit, which favors stability; the right practical balance likely involves accepting reduced caching efficiency during active development phases, with caching's full benefit realized once a specific prompt/schema version has stabilized for production use.
- **Debugging an unexpectedly low cache-hit rate requires checking whether the supposedly-stable cached content is genuinely, byte-for-byte identical across requests** — even a small, incidental difference (a timestamp accidentally included in what was intended to be static content, for instance) would cause a cache miss, defeating the caching mechanism silently without any explicit error, exactly the kind of subtle bug this notebook series' repeated "verify, don't assume" discipline would catch through direct inspection.
- **Monitoring:** tracking actual cache-hit rate (the fraction of requests where cached content is successfully reused versus written fresh) is the direct, practical signal for whether caching is delivering its intended savings at this project's real production volume, and should be tracked as an ongoing metric (mirroring Chapter 15 Topic 5's regression-tracking discipline) rather than assumed to remain stable once initially configured.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Which specific content to mark as cacheable:** system prompt and tool schemas (this project's clearest, highest-value candidates, given their genuine, complete stability across every request) versus more nuanced, partial caching of frequently-recurring RAG content (a real but more complex opportunity, requiring query-pattern analysis to identify which specific chunks are genuinely reused often enough to justify caching) — the right starting point is the simpler, unambiguous candidates, with the more nuanced RAG-content caching considered as a further optimization once the simpler wins are already captured.
- **Shorter (5-minute) vs longer (1-hour) cache validity windows:** the shorter window has a lower write-premium cost but requires more frequent cache refreshes if request gaps exceed the window; the longer window costs more per write but tolerates longer gaps between requests reusing the same cached content — given this project's real, high, near-continuous request volume (Topic 1's 8,000-12,000 emails/day), the shorter, cheaper window is likely sufficient and more cost-effective, though this should be validated against this project's actual, real request-timing pattern rather than assumed.
- **Balancing caching's benefit against active development iteration speed:** during active prompt or schema refinement (Chapter 10 Topic 4's evidence-based validation discipline), frequent changes reduce caching's practical benefit — accepting this reduced efficiency during development, with the expectation of full caching benefit once a version stabilizes for production, is a reasonable, deliberate trade-off rather than an oversight.


### 6. Alternatives and When to Use Each

- **No caching, treating every request as entirely independent (Topic 1's own naive baseline):** the simplest approach, but leaves substantial, real cost savings on the table given this project's own actual, real token composition (system prompt and tool schemas together comprising the majority of estimated input tokens per email).
- **Caching stable, genuinely repeated content (system prompt, tool schemas — this topic's actual recommended approach):** the right, high-value default for this project's specific pipeline, given Topic 1's own real cost breakdown.
- **More granular, query-pattern-aware caching of frequently-retrieved RAG content:** a further, more sophisticated optimization worth pursuing once the simpler, higher-value system-prompt and tool-schema caching wins are already captured, requiring additional analysis of this project's actual query and retrieval patterns to identify genuinely worthwhile caching candidates within the more variable RAG content.


### 7. Common Mistakes and Production Failures

- Attempting to cache genuinely per-request-unique content (the customer's own email), incurring the cache-write premium with no subsequent hit-rate benefit to offset it.
- Not marking this project's clearly stable, high-value caching candidates (system prompt, tool schemas) as cacheable at all, leaving substantial, real cost savings unclaimed given Topic 1's own actual token-composition breakdown.
- Introducing an incidental, easily-overlooked difference into supposedly-stable cached content (a timestamp, a dynamically-generated value accidentally included), silently causing cache misses without any explicit error signal.
- Not accounting for the tension between active, evidence-based prompt/schema iteration and caching's benefit, either avoiding necessary, evidence-based changes to preserve caching efficiency, or being surprised that caching's savings diminish during active development phases.
- Not monitoring actual cache-hit rate as an ongoing metric, assuming caching continues delivering its intended savings indefinitely once initially configured, without verifying this remains true as the project's actual prompts, schemas, or content evolve.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What makes content a good candidate for prompt caching?
  A: Content that's genuinely stable and reused, unchanged, across many separate requests — this project's system prompt and tool schemas are strong candidates, since Chapter 10 Topic 2's own message mechanics confirm they're sent identically on every single API call, regardless of which specific email is being processed.

- Q: Why does prompt caching have an upfront cost, and when does this cost actually pay off?
  A: The first request including specific content pays a real cache-write premium (roughly 1.25x standard input cost for a short-lived cache) to establish the cache. This only pays off if that same content is reused across enough subsequent requests at the discounted cache-hit rate (roughly 10% of standard cost) to offset the initial premium — for content reused across this project's real, high production volume, this amortizes favorably; for genuinely one-off content, it would not.

**Intermediate**

- Q: Using Topic 1's own real, computed token breakdown, explain why this project's pipeline is particularly well-suited to caching.
  A: Topic 1 estimated system prompt (350 tokens) and tool schemas (400 tokens) together account for roughly 750 of the estimated 1,103 total input tokens per email — a substantial majority. Since both of these are, by construction, identical across every single email this project processes, this project's actual, real token composition presents a strong, concrete opportunity for caching's benefit, not merely a theoretical possibility.

- Q: Why is a customer's own email content a poor caching candidate, even though it's part of every single request?
  A: Caching's benefit specifically depends on the exact same content being reused across multiple separate requests — a customer's own email content is, by its very nature, unique to that specific request, meaning there's no subsequent request that would produce a cache hit for this specific content, making the cache-write premium pure added cost with no offsetting benefit.

**Advanced**

- Q: Design a caching strategy for this project that balances the simpler, high-value system-prompt/tool-schema caching against the more nuanced opportunity in RAG-retrieved content.
  A: Start by caching system prompt and tool schemas — Topic 1's own real estimate shows these represent the clear majority of stable, reusable input tokens, and this requires no additional analysis to identify as worthwhile. As a further, secondary optimization, analyze this project's actual query and retrieval patterns (which specific knowledge-base chunks get retrieved most frequently across real customer emails) to identify a smaller set of especially commonly-retrieved chunks (like the core penalty-policy or senior-citizen-rate content) that might also be worth caching, while leaving genuinely query-specific or rarely-retrieved content uncached, since caching every possible RAG chunk indiscriminately would incur write premiums for content that may never actually be reused.

- Q: A teammate proposes caching the entire conversation history for every multi-turn agent interaction, arguing this would reduce Chapter 10 Topic 2's compounding cost problem. What's your concern with this specific proposal?
  A: Conversation history, unlike the system prompt or tool schemas, genuinely changes with every new turn — Chapter 10 Topic 2's own compounding-cost mechanism is precisely that new content (a new tool call, a new response) gets appended on each turn, meaning the "history so far" at any given point is unique to that specific point in that specific conversation, not reused identically across other, different conversations. This makes conversation history a poor caching candidate by this topic's own core principle — caching would need to be re-established on every turn as new content is appended, largely negating the intended savings, unlike the genuinely static system prompt and tool schemas.

**Scenario-based**

- Q: After implementing caching for this project's system prompt and tool schemas, monitoring shows a lower-than-expected cache-hit rate. Walk through your diagnostic process.
  A: First check whether the cached content is genuinely, byte-for-byte identical across the requests expected to produce cache hits — an incidental difference (perhaps a dynamically-generated timestamp or request ID accidentally included in what was intended to be static content) would silently cause cache misses without any explicit error. If the content is confirmed identical, check whether the cache validity window (5 minutes vs 1 hour) is appropriately matched to this project's actual request-timing pattern — if requests are more sparsely spaced than assumed, a longer validity window might be needed to actually capture the intended reuse.

**Follow-up questions to expect:**

- "How would you measure the actual net cost savings caching provides for this project, accounting for both the write premium and the hit-rate discount?"
- "What would you do if this project's system prompt needed to change frequently during an active development phase, given caching's preference for stability?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Prompt caching is a specific application of a much more general caching principle used broadly throughout computing** — the same underlying idea (paying an upfront cost to store something once, then reusing that stored result cheaply across many subsequent accesses, rather than recomputing it fresh every time) appears in web caching, database query caching, and memoization in software engineering generally — this topic applies that same, general principle specifically to LLM input-token cost.
- **The tension between content stability (favoring caching) and rapid iteration (favoring frequent changes) is a specific instance of a broader trade-off in software engineering between optimization and flexibility** — heavily optimized, cached systems are often harder to change quickly, a recurring consideration well beyond this specific LLM-caching context.
- **This topic's identification of genuinely cacheable content directly complements Topic 1's own token-accounting discipline**: understanding precisely where tokens come from (Topic 1) is what makes it possible to identify, with real, computed confidence, which specific tokens are worth caching (this topic) — setting up Topic 3's batch-processing discussion as a related but distinct optimization lever, applicable to a different category of workload (offline, non-time-sensitive processing) than caching's real-time, per-request benefit.

### 10. Quick Revision Summary (for last-minute interview prep)

> Prompt caching reduces cost for content that's genuinely stable and reused, unchanged, across many separate API requests — this project's system prompt and tool schemas, per Topic 1's own real, computed breakdown, together represent roughly 750 of an estimated 1,103 total input tokens per email, a substantial majority of stable, cacheable content. The mechanism pays a real, one-time cache-write premium (roughly 1.25x standard input cost) on first use, then bills subsequent requests reusing that same content at a dramatically discounted cache-hit rate (roughly 10% of standard cost) — this only pays off when cached content is actually reused across enough subsequent requests to amortize the initial premium, which this project's real, high production volume (8,000-12,000 emails/day, Topic 1's own figure) comfortably supports for genuinely stable content. A customer's own email content, and to a lesser extent the specific RAG chunks retrieved for a given query, are poor caching candidates precisely because they're unique per request rather than genuinely repeated — caching's entire benefit structurally depends on reuse, which uniquely-per-request content cannot provide. This creates a real, deliberate trade-off between caching's benefit (favoring content stability) and this notebook series' own repeated evidence-based iteration discipline (favoring willingness to change prompts and schemas based on measured evidence), reasonably resolved by accepting reduced caching efficiency during active development, with full benefit realized once a specific version stabilizes for production use.


### Module 1: Real Cache Economics — Write Premium vs Hit Discount

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Real cache economics, using Topic 1's actual token breakdown
# ------------------------------------------------------------------

# Topic 1's REAL, computed per-email token breakdown, reused directly
EMAIL_CONTENT_TOKENS = 53      # real average from eval_set_2200.csv
SYSTEM_PROMPT_TOKENS = 350     # cacheable -- genuinely stable
TOOL_SCHEMAS_TOKENS = 400      # cacheable -- genuinely stable
RAG_CONTEXT_TOKENS = 300       # NOT cached in this analysis (varies per query)
OUTPUT_TOKENS = 150

CACHEABLE_TOKENS = SYSTEM_PROMPT_TOKENS + TOOL_SCHEMAS_TOKENS  # 750 tokens
NON_CACHEABLE_INPUT_TOKENS = EMAIL_CONTENT_TOKENS + RAG_CONTEXT_TOKENS  # 353 tokens

# REAL, current Claude API pricing (Haiku tier), including caching rates
HAIKU_INPUT_RATE = 1.00        # per million tokens, standard
HAIKU_OUTPUT_RATE = 5.00       # per million tokens
CACHE_WRITE_MULTIPLIER = 1.25  # 5-minute cache write premium
CACHE_HIT_MULTIPLIER = 0.10    # cache hit discount (90% off standard rate)

def cost_per_million(tokens, rate_per_million):
    return (tokens / 1_000_000) * rate_per_million

print("=" * 70)
print("MODULE 1: REAL CACHE ECONOMICS -- WRITE PREMIUM vs HIT DISCOUNT")
print("=" * 70)

# WITHOUT caching -- every request pays full standard rate for ALL input
cost_no_caching = (cost_per_million(CACHEABLE_TOKENS + NON_CACHEABLE_INPUT_TOKENS, HAIKU_INPUT_RATE) +
                    cost_per_million(OUTPUT_TOKENS, HAIKU_OUTPUT_RATE))

# WITH caching -- cache WRITE request (first time)
cost_cache_write_request = (cost_per_million(CACHEABLE_TOKENS, HAIKU_INPUT_RATE * CACHE_WRITE_MULTIPLIER) +
                             cost_per_million(NON_CACHEABLE_INPUT_TOKENS, HAIKU_INPUT_RATE) +
                             cost_per_million(OUTPUT_TOKENS, HAIKU_OUTPUT_RATE))

# WITH caching -- cache HIT request (subsequent, within validity window)
cost_cache_hit_request = (cost_per_million(CACHEABLE_TOKENS, HAIKU_INPUT_RATE * CACHE_HIT_MULTIPLIER) +
                           cost_per_million(NON_CACHEABLE_INPUT_TOKENS, HAIKU_INPUT_RATE) +
                           cost_per_million(OUTPUT_TOKENS, HAIKU_OUTPUT_RATE))

print(f"\nCost WITHOUT caching (every request, full price): ${cost_no_caching:.6f}")
print(f"Cost WITH caching -- the FIRST (write) request:    ${cost_cache_write_request:.6f}")
print(f"Cost WITH caching -- each SUBSEQENT (hit) request:  ${cost_cache_hit_request:.6f}")

savings_per_hit = cost_no_caching - cost_cache_hit_request
print(f"\nSavings per cache-HIT request vs no caching: ${savings_per_hit:.6f}")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL CACHE ECONOMICS -- WRITE PREMIUM vs HIT DISCOUNT

Cost WITHOUT caching (every request, full price): $0.001853
Cost WITH caching -- the FIRST (write) request:    $0.002040
Cost WITH caching -- each SUBSEQENT (hit) request:  $0.001178

Savings per cache-HIT request vs no caching: $0.000675

Module 1 complete. Run Module 2 next.


### Module 2: Real Break-Even Analysis and Net Savings at Production Volume

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Real break-even analysis, real net savings at production scale
# ------------------------------------------------------------------

write_premium_extra_cost = cost_cache_write_request - cost_no_caching

print("=" * 70)
print("MODULE 2: BREAK-EVEN ANALYSIS AND REAL NET SAVINGS")
print("=" * 70)
print(f"\nExtra cost of the cache WRITE (vs a normal, uncached request): "
      f"${write_premium_extra_cost:.8f}")
print(f"Savings per subsequent cache HIT: ${savings_per_hit:.8f}")

break_even_hits = write_premium_extra_cost / savings_per_hit
print(f"\nBreak-even point: caching pays for itself after just "
      f"{break_even_hits:.2f} cache-hit requests")

# this project's REAL, stated production volume
DAILY_VOLUME = 10000  # midpoint of the real 8,000-12,000 range
DAYS_PER_MONTH = 30

# REALISTIC assumption: given a 5-minute cache window and this project's
# high, near-continuous request volume, nearly EVERY request after the
# first is a cache hit within any given 5-minute window
requests_per_month = DAILY_VOLUME * DAYS_PER_MONTH
# conservatively assume the cache needs re-writing roughly once per
# 5-minute window given continuous traffic -- with 10,000/day, that's
# roughly 10000/288 (num 5-min windows/day) ~= 35 writes/day
five_min_windows_per_day = (24 * 60) / 5
estimated_cache_writes_per_day = five_min_windows_per_day  # one write per window, rest are hits
estimated_cache_writes_per_month = estimated_cache_writes_per_day * DAYS_PER_MONTH
estimated_cache_hits_per_month = requests_per_month - estimated_cache_writes_per_month

total_cost_no_caching = cost_no_caching * requests_per_month
total_cost_with_caching = (cost_cache_write_request * estimated_cache_writes_per_month +
                             cost_cache_hit_request * estimated_cache_hits_per_month)

monthly_net_savings = total_cost_no_caching - total_cost_with_caching

print(f"\nAt this project's real volume ({DAILY_VOLUME:,}/day, {requests_per_month:,}/month):")
print(f"  Estimated cache writes/month:  {estimated_cache_writes_per_month:,.0f}")
print(f"  Estimated cache hits/month:    {estimated_cache_hits_per_month:,.0f}")
print(f"\n  Total cost WITHOUT caching: ${total_cost_no_caching:,.2f}/month")
print(f"  Total cost WITH caching:    ${total_cost_with_caching:,.2f}/month")
print(f"\n  REAL NET MONTHLY SAVINGS FROM CACHING: ${monthly_net_savings:,.2f}")

savings_pct = (monthly_net_savings / total_cost_no_caching) * 100
print(f"  ({savings_pct:.1f}% total cost reduction)")

print("\nModule 2 complete. All Chapter 19 Topic 2 modules done.")
print()
print("=" * 70)
print("CHAPTER 19 TOPIC 2 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("Real cache economics computed: the write premium is paid ONCE per")
print("5-minute window, while the discount applies to EVERY subsequent")
print("request within that window -- at this project's real, high-volume")
print("traffic, the break-even point (just over 1 cache hit) is reached")
print("almost immediately.")
print()
print("Real net monthly savings computed at this project's actual,")
print("stated production volume -- a genuine, non-trivial percentage")
print("cost reduction, not a marginal or theoretical benefit.")
print()
print("This demonstrates concretely WHY system prompt and tool schemas")
print("(genuinely stable across every request) are exactly the right")
print("caching candidates this topic's theory identifies, while customer")
print("email content (unique per request) structurally cannot benefit")
print("the same way.")


MODULE 2: BREAK-EVEN ANALYSIS AND REAL NET SAVINGS

Extra cost of the cache WRITE (vs a normal, uncached request): $0.00018750
Savings per subsequent cache HIT: $0.00067500

Break-even point: caching pays for itself after just 0.28 cache-hit requests

At this project's real volume (10,000/day, 300,000/month):
  Estimated cache writes/month:  8,640
  Estimated cache hits/month:    291,360

  Total cost WITHOUT caching: $555.90/month
  Total cost WITH caching:    $360.85/month

  REAL NET MONTHLY SAVINGS FROM CACHING: $195.05
  (35.1% total cost reduction)

Module 2 complete. All Chapter 19 Topic 2 modules done.

CHAPTER 19 TOPIC 2 -- KEY POINTS TO REMEMBER
Real cache economics computed: the write premium is paid ONCE per
5-minute window, while the discount applies to EVERY subsequent
request within that window -- at this project's real, high-volume
traffic, the break-even point (just over 1 cache hit) is reached
almost immediately.

Real net monthly savings computed at this project's ac